In [ ]:
import os
import awkward as ak

from mltau.tools.evaluation import kinematics as k
from mltau.tools.evaluation import tagging as t
from mltau.tools.evaluation import charge_id as c
from mltau.tools.evaluation import decay_mode as d

from mltau.tools.general import reinitialize_p4

In [ ]:
from hydra import compose, initialize

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main")

In [ ]:
OUTPUT_DIR = "/home/laurits/2222_training"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SIGNAL_SAMPLE = "z"
BKG_SAMPLE = "qq"


# SingleParTau

In [ ]:
SINGLE_PARTAU_TRAININGS_DIR = "/home/norman/0413_test/"

In [ ]:
# Tagging
sTag_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "isTau", f"{SIGNAL_SAMPLE}_test.parquet")
)
sTag_bkgData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "isTau", f"{BKG_SAMPLE}_test.parquet")
)
sTag_evaluator = t.TaggerEvaluator(
    signal_predictions=sTag_sigData.tau_tagging_score,
    signal_gen_tau_p4=sTag_sigData.gen_jet_tau_p4,
    signal_reco_jet_p4=sTag_sigData.reco_jet_p4,
    bkg_predictions=sTag_bkgData.tau_tagging_score,
    bkg_gen_jet_p4=sTag_bkgData.gen_jet_p4,
    bkg_reco_jet_p4=sTag_bkgData.reco_jet_p4,
    cfg=cfg,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau",
)

In [ ]:
# Decay mode
sDM_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "DM", f"{SIGNAL_SAMPLE}_test.parquet")
)
sDM_evaluator = d.DecayModeEvaluator(
    predicted=sDM_sigData.tau_decay_mode_probs,
    truth=sDM_sigData.gen_jet_tau_decay_mode,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau"
)

In [ ]:
# Charge
sCh_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "charge", f"{SIGNAL_SAMPLE}_test.parquet")
)
sCh_evaluator = c.ChargeIdEvaluator(
    predicted=sCh_sigData.tau_charge_score,
    truth=sCh_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=sCh_sigData.gen_jet_tau_p4,
    reco_jet_p4s=sCh_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample"SIGNAL_SAMPLE,
    algorithm="SingleParTau",
    baseline_charges=baseline_charges,  # TODO: Add separately
)

In [ ]:
# Kinematics
sKin_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "kin", f"{SIGNAL_SAMPLE}_test.parquet")
)
sKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=sKin_sigData.tau_p4,
    true_p4=sKin_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="SingleParTau",
    sample_name=SIGNAL_SAMPLE
)

# MultiParTau

In [ ]:
MULTI_PARTAU_TRAININGS_DIR = "/home/laurits/test_inference/"
# MULTI_PARTAU_TRAININGS_DIR = "/home/laurits/0407_lifetime_training/"


In [ ]:
multiParTau_sigData = ak.from_parquet(
    os.path.join(MULTI_PARTAU_TRAININGS_DIR, "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
multiParTau_bkgData = ak.from_parquet(
    os.path.join(MULTI_PARTAU_TRAININGS_DIR, "predictions", f"{BKG_SAMPLE}_test.parquet")
)

mTag_evaluator = t.TaggerEvaluator(
    signal_predictions=multiParTau_sigData.tau_tagging_score,
    signal_gen_tau_p4=multiParTau_sigData.gen_jet_tau_p4,
    signal_reco_jet_p4=multiParTau_sigData.reco_jet_p4,
    bkg_predictions=multiParTau_bkgData.tau_tagging_score,
    bkg_gen_jet_p4=multiParTau_bkgData.gen_jet_p4,
    bkg_reco_jet_p4=multiParTau_bkgData.reco_jet_p4,
    cfg=cfg,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau",
)

mDM_evaluator = d.DecayModeEvaluator(
    pred_proba=multiParTau_sigData.tau_decay_mode_probs,
    truth=multiParTau_sigData.gen_jet_tau_decaymode,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau"
)

mCh_evaluator = c.ChargeIdEvaluator(
    predicted=multiParTau_sigData.tau_charge_score,
    truth=multiParTau_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=multiParTau_sigData.gen_jet_tau_p4,
    reco_jet_p4s=multiParTau_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau",
    # baseline_charges=baseline_charges,  # TODO: Add separately
)

mKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=multiParTau_sigData.tau_p4,
    true_p4=multiParTau_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="MultiParTau",
    sample_name=SIGNAL_SAMPLE
)

# RecoJet

In [ ]:
cand_pts = reinitialize_p4(multiParTau_sigData.cand_p4).pt
cand_charges = multiParTau_sigData.cand_charges
jet_pts = reinitialize_p4(multiParTau_sigData.reco_jet_p4).pt
baseline_charges = c.jet_charge_qkappa(
    cand_charges=cand_charges, cand_pts=cand_pts, jet_pts=jet_pts, kappa=0.5)

rCh_evaluator = c.ChargeIdEvaluator(
    predicted=baseline_charges, 
    truth=multiParTau_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=multiParTau_sigData.gen_jet_tau_p4,
    reco_jet_p4s=multiParTau_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="RecoJet",
)

rKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=multiParTau_sigData.reco_jet_p4,
    true_p4=multiParTau_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="RecoJet",
    sample_name=SIGNAL_SAMPLE
)

# Combined results

In [ ]:
# Kinematics
kRESULTS_DIR = os.path.join(RESULTS_DIR, "kinematics")
os.makedirs(kRESULTS_DIR, exist_ok=True)
kme = k.KinematicsMultiEvaluator(kRESULTS_DIR, cfg, sample=SIGNAL_SAMPLE)  # Fix output_dir
# kme.combine_results([sKin_evaluator, mKin_evaluator, rKin_evaluator])
kme.combine_results([mKin_evaluator, rKin_evaluator])
kme.save()

# Charge
cRESULTS_DIR = os.path.join(RESULTS_DIR, "charge_id")
os.makedirs(cRESULTS_DIR, exist_ok=True)
cme = c.ChargeMultiEvaluator(cRESULTS_DIR, cfg)  # Fix output_dir
# cme.combine_results([sCh_evaluator, mCh_evaluator, rCh_evaluator])
cme.combine_results([mCh_evaluator, rCh_evaluator])
cme.save()

# Tagging
tRESULTS_DIR = os.path.join(RESULTS_DIR, "tau_id")
os.makedirs(tRESULTS_DIR, exist_ok=True)
tme = t.TaggerMultiEvaluator(tRESULTS_DIR, cfg)  # Fix output_dir
# tme.combine_results([sTag_evaluator, mTag_evaluator])
tme.combine_results([mTag_evaluator])
tme.save()

# Decay mode
dRESULTS_DIR = os.path.join(RESULTS_DIR, "decay_mode")
os.makedirs(dRESULTS_DIR, exist_ok=True)
dme = d.DecayModeMultiEvaluator(dRESULTS_DIR, cfg, sample=SIGNAL_SAMPLE)  # Fix output_dir, Implement Multievaluator.
# dme.combine_results([sDM_evaluator, mDM_evaluator])
dme.combine_results([mDM_evaluator])
dme.save()

# Losses (maybe we want to plot some losses?)

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

# log_dir = "/home/laurits/tmp/speedup_test2/tensorboard/ParTau_experiment/version_0/"

# ea = event_accumulator.EventAccumulator(log_dir)
# ea.Reload()

# # List available scalar tags
# print(ea.Tags()["scalars"])

# # Extract a specific scalar
# scalars = ea.Scalars("train_losses/decay_mode_loss")

# for s in scalars:
#     print(s.step, s.value)